# 1 · Diagnose and evaluate

**Can a clear answer still be wrong?** The item is not defective. The customer asks for a full refund 45 days after buying it.

Current rule: full refund through day 30, store credit after day 30. Support must review defective items.


In [ ]:
import importlib
import os, sys, json
from pathlib import Path

# Find the repository folder in local Jupyter or Databricks.
start = Path(os.environ.get("WORKSHOP_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "west_workshop" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set WORKSHOP_PROJECT_ROOT to the workshop repository folder, then restart this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
os.chdir(ROOT)
MODE = os.environ.get("WORKSHOP_MODE", "saved")  # Choose live mode to make new model calls.


In [ ]:
from IPython.display import display, HTML, Code
from west_workshop.repair_workshop import notebook_session
from west_workshop.mlflow_view import session_header, inspection_card

session = notebook_session(MODE)
display(HTML(session_header(session)))


## Code view 1: one business check

Does the answer name the right outcome: full refund, store credit or support review? This label check is one of five code checks in `deterministic_stack`. It does not judge the explanation.


In [ ]:
import inspect
from west_workshop.runtime import policy_decision_value

display(Code(inspect.getsource(policy_decision_value), language="python"))


## Code view 2: evaluate the application

`data` holds the test cases. `predict_fn` runs the app and records its steps. `scorers` lists the checks for each answer.
Saved mode reads recorded results. Live mode runs new evaluations and reuses completed results from the same output folder.


In [ ]:
if session.needs("deployed"):
    import mlflow
    with session.evaluation("deployed") as run:
        result = mlflow.genai.evaluate(
            data=run.data, predict_fn=run.predict_fn, scorers=run.scorers,
        )
        session.record_evaluation("deployed", result, run)


In [ ]:
deployed = session.recorded("deployed")
if not deployed["complete"]:
    raise RuntimeError("This evaluation is incomplete. Read the saved error before trying again in a new output folder.")


## Inspect in MLflow: request → policy → check results

A trace shows what happened during one request. Open `day_45_opening`. Which policy did the assistant receive? Which check caught the wrong answer?

If you cannot open these results in MLflow, expand the recorded case below.


In [ ]:
display(HTML(inspection_card(session, "deployed", ["day_45_opening"])))


Next: **02_compare_and_decide.ipynb**. Test how answers are graded, then check the repair.
